# Controlling Thinking Speed

Replicates **"Controlling Thinking Speed in Reasoning Models"** ([arXiv:2507.03704](http://arxiv.org/abs/2507.03704)) on DeepSeek-R1-Distill-Qwen-1.5B, end to end in one engine, following the paper's stimulus design (Appendix A.1):

1. **Construction** — fast-thinking traces (reasoning primed with "To") and slow-thinking traces (default reasoning) are sampled on MATH500 problems, filtered to pairs where both reach the correct answer, and turned into initial-segment stimuli; the final-token hidden states feed the paper's symmetrized pair-difference PCA (`MATH500.gguf`).
2. **Steering** — adding the slow→fast direction at layers 19–27 during generation enhances fast thinking, cutting the mean reasoning length over 100 MATH-500 problems (`math500.json`).

Uses the current EasySteer steering API (`SteeringSpec` / `VectorSpec` / `ApplySpec`).

**Execution note:** Outputs are cleared after the API migration. Run the notebook from top to bottom to obtain results for your environment.


In [ ]:
import json
import os

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("VLLM_LOGGING_LEVEL", "WARNING")  # quiet engine boot logs

from vllm import LLM, SamplingParams
from vllm.steer_vectors import ApplySpec, SteeringSpec, VectorSpec

MODEL = os.environ.get("EASYSTEER_MODEL", "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B")  # deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B

# One engine serves both construction (capture) and steering.
llm = LLM(
    model=MODEL,
    tensor_parallel_size=int(os.environ.get("EASYSTEER_TP", "1")),
    enable_steer_vector=True,
    steer_algorithms=["direct"],
)
tok = llm.get_tokenizer()

with open("math500_problems.json", encoding="utf-8") as f:
    problems = json.load(f)

INSTRUCTION = "Please reason step by step, and put your final answer within \\boxed{}."


def prompt_ids(problem, primer=""):
    text = tok.apply_chat_template(
        [{"role": "user", "content": f"{INSTRUCTION}\n{problem}"}],
        tokenize=False,
        add_generation_prompt=True,  # ends with "<｜Assistant｜><think>\n"
    )
    return tok(text + primer, add_special_tokens=False).input_ids

## Vector construction

### Sample fast and slow traces

Fast thinking is elicited by priming the thought with "To" (the paper's trigger word); slow thinking is the model's default reasoning. Greedy decoding keeps the construction reproducible.

In [ ]:
params = SamplingParams(temperature=0, max_tokens=8192, skip_special_tokens=False)

fast_out = llm.generate([{"prompt_token_ids": prompt_ids(p["problem"], "To")}
                         for p in problems], params, use_tqdm=False, steering=False)
slow_out = llm.generate([{"prompt_token_ids": prompt_ids(p["problem"])}
                         for p in problems], params, use_tqdm=False, steering=False)

### Filter to valid pairs

The paper keeps only stimulus pairs whose responses **both end with correct answers**; truncated generations never reach an answer and are dropped by the same test.

In [ ]:
import re


def normalize(ans):
    return re.sub(r"\\left|\\right|\\!|\s+", "", ans)


def boxed_answer(text):
    start = text.rfind("\\boxed{")
    if start == -1:
        return None
    i, depth = start + len("\\boxed{"), 1
    for j in range(i, len(text)):
        depth += {"{": 1, "}": -1}.get(text[j], 0)
        if depth == 0:
            return text[i:j]
    return None


pairs = []
for prob, fast, slow in zip(problems, fast_out, slow_out):
    fast_text = "To" + fast.outputs[0].text
    slow_text = slow.outputs[0].text
    ok = all(
        out.outputs[0].finish_reason == "stop"
        and boxed_answer(text) is not None
        and normalize(boxed_answer(text)) == normalize(prob["answer"])
        for out, text in ((fast, fast_text), (slow, slow_text))
    )
    if ok:
        pairs.append((prob["problem"], fast_text, slow_text))

print(f"valid pairs: {len(pairs)} / {len(problems)}")
assert len(pairs) >= 4, "too few valid pairs; raise max_tokens or add problems"

### Build stimuli from the initial thought segments

Per Appendix A.1, using whole traces (or only the first token) weakens the direction: the fast stimulus keeps the first 2 `\n\n`-separated steps of its thought, and the slow stimulus is truncated to the step whose cumulative length is nearest to its paired fast stimulus.

In [ ]:
def thought(text):
    return text.split("</think>")[0]


def first_steps(text, n):
    return "\n\n".join(thought(text).split("\n\n")[:n])


def nearest_length_steps(text, target_len):
    steps = thought(text).split("\n\n")
    best = steps[0]
    for n in range(1, len(steps) + 1):
        candidate = "\n\n".join(steps[:n])
        if abs(len(candidate) - target_len) <= abs(len(best) - target_len):
            best = candidate
    return best


stimuli_fast = []
stimuli_slow = []
for problem, fast_text, slow_text in pairs:
    base = tok.apply_chat_template(
        [{"role": "user", "content": f"{INSTRUCTION}\n{problem}"}],
        tokenize=False,
        add_generation_prompt=True,
    )
    fast_stim = first_steps(fast_text, 2)
    stimuli_fast.append(base + fast_stim)
    stimuli_slow.append(base + nearest_length_steps(slow_text, len(fast_stim)))

In [ ]:
from easysteer.capture import capture
from vllm.capture import SelectSpec

# The paper reads the hidden state at the stimulus's final token.
result = capture(
    llm,
    [{"prompt_token_ids": tok(s, add_special_tokens=False).input_ids}
     for s in stimuli_fast + stimuli_slow],
    select=SelectSpec(prompt_positions=[-1]),
    steering=False,
)

In [ ]:
from easysteer.extraction import extract

# Allow the exact estimator to process up to 500 positive/negative pairs.
# Retain exact pair-centered PCA, with each positive paired to the
# corresponding negative. Pool selected tokens before fitting PCA.
labels = [True] * len(pairs) + [False] * len(pairs)
control_vector = extract(
    result,
    labels,
    method="pca",
    variant="center",
    max_working_bytes=512 * 1024**2,
    token_pos=-1,
    normalize=False,
)
control_vector.export_gguf("MATH500.gguf")


## Steering

In [ ]:
# Baseline: no steering. As in the experiment section, evaluate on
# 100 MATH-500 problems and report only the mean generated length —
# report the observed aggregate for this run.
with open("math500.json", encoding="utf-8") as f:
    eval_problems = [x["problem"] for x in json.load(f)][:100]
eval_ids = [{"prompt_token_ids": prompt_ids(p)} for p in eval_problems]
params = SamplingParams(temperature=0, max_tokens=8192, skip_special_tokens=False)


def mean_tokens(outputs):
    return sum(len(o.outputs[0].token_ids) for o in outputs) / len(outputs)


baseline = mean_tokens(llm.generate(eval_ids, params, use_tqdm=False, steering=False))
print(f"Baseline mean tokens: {baseline:.0f}")

In [ ]:
# Fast thinking: positive scale on the slow->fast direction at the
# paper's control layers (19-27).
steering_fast = SteeringSpec(vectors=[
    VectorSpec(
        source="MATH500.gguf",
        scale=4.0,
        layers=list(range(19, 28)),
        apply=ApplySpec(prompt="all", generation="all"),
    ),
])
fast = mean_tokens(llm.generate(eval_ids, params, steering=steering_fast,
                                use_tqdm=False))
print(f"Fast mean tokens: {fast:.0f} ({(fast / baseline - 1) * 100:+.0f}%)")